<a href="https://colab.research.google.com/github/FaKeBx/omnivoice/blob/main/docs/OmniVoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [1]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/162.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [2]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able 

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

## 4. Expondo como API (para Next.js)

Nesta seção, vamos configurar um servidor FastAPI e usar o `ngrok` para criar um túnel público. Assim, seu projeto Next.js poderá enviar requisições POST para este notebook.

In [3]:
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart

In [ ]:
import nest_asyncio
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
from pyngrok import ngrok
import uvicorn
import shutil
import os
import asyncio

app = FastAPI()

@app.post("/clone")
async def clone_voice(text: str = Form(...), file: UploadFile = File(...)):
    # Salva o arquivo temporário enviado pelo Next.js
    temp_ref = "temp_ref.wav"
    with open(temp_ref, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    # Gera o áudio usando o modelo carregado no notebook
    audio = model.generate(text=text, ref_audio=temp_ref)

    # Salva o resultado
    output_path = "api_output.wav"
    sf.write(output_path, audio[0], 24000)

    return FileResponse(output_path, media_type="audio/wav")

# Token do ngrok
auth_token = "33q8px58A5ONVlYM2GFKbSm4fcV_4WgGw6ZT4eGx1LQRv8Bt7"
if auth_token:
    ngrok.set_auth_token(auth_token)

# Inicia o túnel
public_url = ngrok.connect(8000).public_url
print(f"URL Pública da API: {public_url}")
print(f"Endpoint para o Next.js: {public_url}/clone")

# Aplica o nest_asyncio para permitir uvicorn dentro do notebook
nest_asyncio.apply()

# Configura e roda o servidor de forma assíncrona compatível com Colab
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

URL Pública da API: https://benmost-norman-ultrasonically.ngrok-free.dev
Endpoint para o Next.js: https://benmost-norman-ultrasonically.ngrok-free.dev/clone


INFO:     Started server process [4356]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
/usr/local/lib/python3.12/dist-packages/omnivoice/utils/audio.py:63: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(audio_path, sr=None, mono=False)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for y

INFO:     2804:14d:5e82:9290:15ed:fe5d:61e4:4eff:0 - "POST /clone HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/omnivoice/utils/audio.py:63: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(audio_path, sr=None, mono=False)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


INFO:     2804:14d:5e82:9290:15ed:fe5d:61e4:4eff:0 - "POST /clone HTTP/1.1" 200 OK


...

INFO:     98.93.129.75:0 - "POST /clone HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/omnivoice/utils/audio.py:63: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(audio_path, sr=None, mono=False)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


INFO:     98.93.129.75:0 - "POST /clone HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/omnivoice/utils/audio.py:63: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(audio_path, sr=None, mono=False)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


INFO:     2804:14d:5e82:9290:15ed:fe5d:61e4:4eff:0 - "POST /clone HTTP/1.1" 200 OK
